# Puesta en producción — Predicción del precio de un apartamento nuevo
### Taller de Regresión y Regularización · Grupo B

Este notebook **no reentrena nada**. Carga los artefactos generados por `01_modelado_regresion.ipynb` y predice el precio de un apartamento nuevo.

**Artefactos necesarios** (carpeta `modelado/`):

| Archivo | Contenido |
|---|---|
| `output/modelo_final.joblib` | `Pipeline` de scikit-learn serializado: imputación → `log1p` → escalado → one-hot → regresión lineal → exponencial. Recibe datos crudos y devuelve el precio en pesos. |
| `output/capas_geo.joblib` | Estaciones de TransMilenio, parques, equipamientos y vías, para calcular las distancias Haversine de una ubicación nueva. |
| `output/metadata.json` | Variables del modelo, campos de entrada, métricas y versiones de librerías. |
| `output/ejemplo_entrada.json` | Ejemplo de datos de entrada. |
| `src/preparacion.py`, `src/geo_features.py` | Las mismas funciones de transformación usadas en el entrenamiento. |
| `requirements.txt` | Lista de librerías con versiones. |

**Cómo se transforman las variables armonizadas en un apartamento nuevo** (función `construir_registro`):

- **Antigüedad:** se puede dar como rango (`"Entre 5 y 10 años"`) o como número de años (`7`); en ambos casos se lleva al mismo rango categórico usado en el entrenamiento.
- **Comodidades:** binarias 1/0 (sí/no). Las que no se informen se toman como 0.
- **Administración:** opcional; si falta o está fuera de \$20 mil–\$8 M, el `Pipeline` la imputa con la mediana de entrenamiento.
- **Ubicación:** a partir de `latitud` y `longitud` se calculan las distancias Haversine a TransMilenio, parque, hospitales y vía principal, y los conteos de farmacias y colegios a 500 m. La `localidad` se puede indicar o dejar en `"auto"` (se infiere de las viviendas de entrenamiento más cercanas).

> Si se mueve el notebook, solo hay que ajustar `DIR_MODELADO` en la primera celda.

In [1]:
import json
import sys
from pathlib import Path

import joblib
import pandas as pd
import sklearn

# Carpeta `modelado/` (ajustar esta ruta si el notebook se mueve de lugar)
DIR_MODELADO = Path.cwd() if (Path.cwd() / "output").exists() else Path.cwd() / "modelado"
DIR_OUTPUT = DIR_MODELADO / "output"
sys.path.insert(0, str(DIR_MODELADO.parent))

from modelado.src.geo_features import GeoIndex
from modelado.src.preparacion import construir_registro

pd.set_option("display.max_columns", 100)

## 1. Cargar los artefactos

In [2]:
modelo = joblib.load(DIR_OUTPUT / "modelo_final.joblib")        # Pipeline entrenado
geo = GeoIndex.cargar(DIR_OUTPUT / "capas_geo.joblib")          # capas geográficas
metadata = json.loads((DIR_OUTPUT / "metadata.json").read_text(encoding="utf-8"))

print("Modelo:", metadata["modelo"], "| entrenado el", metadata["fecha_entrenamiento"], "con", f"{metadata['n_train']:,}", "apartamentos")
print("Versión de scikit-learn al entrenar:", metadata["versiones"]["scikit-learn"], "| instalada:", sklearn.__version__)
print(f"Error esperado (prueba): MAE ${metadata['metricas_test']['MAE']/1e6:,.0f} M | MAPE {metadata['metricas_test']['MAPE']:.1f} % | R² {metadata['metricas_test']['R2']:.3f}")
print("\nDatos que debe dar el usuario:")
for c in metadata["campos_formulario"]:
    print(f"  - {c['nombre']:<24} {c['etiqueta']}")
print("\nVariables que se calculan solas con la ubicación:", ", ".join(metadata["variables_calculadas_con_la_direccion"]))

Modelo: Forward (BIC) | entrenado el 2026-11-14 con 8,136 apartamentos
Versión de scikit-learn al entrenar: 1.9.1 | instalada: 1.9.1
Error esperado (prueba): MAE $212 M | MAPE 16.0 % | R² 0.877

Datos que debe dar el usuario:
  - area_m2                  Área (m²)
  - administracion           Administración mensual ($)
  - habitaciones             Habitaciones
  - banos                    Baños
  - parqueaderos             Parqueaderos
  - estrato                  Estrato
  - parqueadero_cubierto     Parqueadero cubierto
  - tiene_deposito           Depósito
  - tiene_terraza            Terraza
  - tiene_estudio            Estudio o biblioteca
  - tiene_cuarto_servicio    Cuarto de servicio
  - tiene_tina               Tina
  - tiene_piscina            Piscina
  - tiene_gimnasio           Gimnasio
  - tiene_bbq                Zona BBQ
  - tiene_zona_infantil      Zona infantil
  - antiguedad_rango         Antigüedad
  - localidad                Localidad

Variables que se calculan sola

## 2. Datos de entrada de un apartamento nuevo

Se lee el ejemplo entregado. Para predecir otro inmueble basta con editar este diccionario (o el archivo `ejemplo_entrada.json`).

In [3]:
apartamento = json.loads((DIR_OUTPUT / "ejemplo_entrada.json").read_text(encoding="utf-8"))
apartamento

{'direccion': 'Calle 94 # 18 - 65, Bogotá',
 'latitud': 4.68055,
 'longitud': -74.05351,
 'area_m2': 88,
 'habitaciones': 2,
 'banos': 2,
 'parqueaderos': 1,
 'estrato': 6,
 'antiguedad_rango': 'Más de 20 años',
 'administracion': 980000,
 'localidad': 'auto',
 'tiene_ascensor': 0,
 'parqueadero_cubierto': 0,
 'tiene_deposito': 1,
 'tiene_balcon': 0,
 'tiene_terraza': 0,
 'tiene_chimenea': 0,
 'tiene_estudio': 1,
 'tiene_cuarto_servicio': 0,
 'tiene_jacuzzi': 0,
 'tiene_tina': 0,
 'tiene_vigilancia': 0,
 'conjunto_cerrado': 0,
 'tiene_piscina': 0,
 'tiene_gimnasio': 1,
 'tiene_salon_comunal': 0,
 'tiene_bbq': 0,
 'tiene_zona_infantil': 0,
 'tiene_zonas_verdes': 0,
 'tiene_sauna_turco': 0,
 'tiene_canchas': 0}

## 3. Aplicar la misma transformación del entrenamiento

`construir_registro` armoniza la antigüedad, valida la administración, convierte las comodidades a 0/1 y calcula las variables de ubicación. El resultado es una fila con las mismas columnas crudas que recibió el `Pipeline` al entrenar; el escalado, la imputación y el one-hot los aplica el propio `Pipeline`.

In [4]:
registro = construir_registro(apartamento, geo)
registro[metadata["variables_modelo"]].T.rename(columns={0: "valor"})

,valor
area_m2,88
administracion,980000
dist_estacion_m,599.8
dist_parque_m,56.8
dist_hospitales_clinicas_m,157.7
dist_via_principal_m,34.4
habitaciones,2
banos,2
parqueaderos,1
estrato,6


## 4. Predicción

In [5]:
precio = float(modelo.predict(registro)[0])
banda = metadata["banda_80"]

print(f"Dirección de referencia : {apartamento.get('direccion', '—')}")
print(f"Localidad               : {registro.loc[0, 'localidad']}")
print(f"Estación más cercana    : {registro.loc[0, 'estacion']} ({registro.loc[0, 'dist_estacion_m']:,.0f} m)")
print(f"Parque más cercano      : {registro.loc[0, 'parque']} ({registro.loc[0, 'dist_parque_m']:,.0f} m)")
print()
print(f"PRECIO ESTIMADO         : ${precio:,.0f}")
print(f"Rango probable (80 %)   : ${precio * banda['factor_inferior']:,.0f}  –  ${precio * banda['factor_superior']:,.0f}")
print(f"Precio por m²           : ${precio / apartamento['area_m2']:,.0f}")

Dirección de referencia : Calle 94 # 18 - 65, Bogotá
Localidad               : CHAPINERO
Estación más cercana    : Calle 100 - Marketmedios (600 m)
Parque más cercano      : CHICO NORTE (57 m)

PRECIO ESTIMADO         : $737,053,010
Rango probable (80 %)   : $569,706,518  –  $952,389,642
Precio por m²           : $8,375,602


## 5. Otro apartamento, digitado a mano

Ejemplo con antigüedad en años y sin administración, para mostrar que la transformación es la misma.

In [6]:
otro = {
    "latitud": 4.7030, "longitud": -74.0420,      # sector Santa Bárbara
    "area_m2": 120, "habitaciones": 3, "banos": 3, "parqueaderos": 2, "estrato": 5,
    "antiguedad": 12,                             # en años -> se convierte a "Entre 10 y 20 años"
    "localidad": "auto",                          # se infiere de la ubicación
    "tiene_deposito": 1, "tiene_terraza": 1, "tiene_gimnasio": 1, "tiene_bbq": 1,
}
registro_otro = construir_registro(otro, geo)
precio_otro = float(modelo.predict(registro_otro)[0])
print(f"Antigüedad armonizada: {registro_otro.loc[0, 'antiguedad_rango']} | Localidad inferida: {registro_otro.loc[0, 'localidad']}")
print(f"PRECIO ESTIMADO: ${precio_otro:,.0f}  (rango 80 %: ${precio_otro * banda['factor_inferior']:,.0f} – ${precio_otro * banda['factor_superior']:,.0f})")

Antigüedad armonizada: Entre 10 y 20 años | Localidad inferida: USAQUEN
PRECIO ESTIMADO: $1,003,230,578  (rango 80 %: $775,448,973 – $1,296,333,368)


## 6. De aquí a la aplicación web

La aplicación de la carpeta `app/` hace exactamente estos mismos pasos detrás de un formulario: convierte la dirección en latitud/longitud (geocodificación), llama a `construir_registro` y a `modelo.predict`, y dibuja en un mapa la ubicación, la estación y el parque más cercanos y los equipamientos en un radio de hasta 500 m.